# Juliet frozen Test 전체 평가

Leakage-disjoint frozen Test 8,337개 전체를 사용해 LR·GBDT·Multi-task MLP Router를 같은 outcome matrix에서 비교합니다. Micro, Expert별, Macro 지표와 비용을 함께 출력합니다.

In [1]:
from pathlib import Path

EVAL_ROOT = Path.cwd().resolve()
if EVAL_ROOT.name != 'Model_Evaluation':
    EVAL_ROOT = (EVAL_ROOT / 'Model_Evaluation').resolve()
CONFIG_PATH = EVAL_ROOT / 'configs' / 'full.toml'
COHORT_CONFIG_PATH = EVAL_ROOT / 'configs' / 'cohort_15837.toml'
ENV_FILE = EVAL_ROOT.parent / '.env'
SELECTED_ARTIFACT = EVAL_ROOT / 'artifacts' / 'juliet_utility_router.pkl'
MAX_CANDIDATES_PER_CASE = 4
HARD_NEGATIVES_PER_CASE = 1
MAX_CONCURRENCY = 1000  # OpenRouter 한도에 맞춰 낮출 수 있음
PATCH_CASE_LIMIT = 0


In [2]:
import json, shutil, sys
sys.path.insert(0, str(EVAL_ROOT / 'src'))
from model_evaluation.config import load_config, load_mapping
from model_evaluation.stages.select_cohort import load_cohort_config, ensure_frozen_index, build_cohort_manifests
from model_evaluation.stages.materialize_dataset import materialize_dataset
from model_evaluation.candidates import cache_candidates
from model_evaluation.workflow import plan_outcome_matrix, collect_outcome_matrix, audit_outcome_matrix, evaluate_utility_router
from model_evaluation.live_evaluation import run_batched_patch_evaluation
from model_evaluation.adapters.llm_security import activate_parent_package
activate_parent_package()
from llm_security.routing import BudgetedUtilityRouter

config = load_config(CONFIG_PATH)
mapping = load_mapping(config.paths.mapping)
cohort_config = load_cohort_config(COHORT_CONFIG_PATH)
COHORT_DIR = EVAL_ROOT / 'work' / 'cohort_15837'
RUN_DIR = EVAL_ROOT / 'work' / 'router_evaluation_full_test'
RESULT_DIR = EVAL_ROOT / 'results' / 'router_evaluation_full_test'
TRAINING_REPORT = EVAL_ROOT / 'results' / 'router_training_stratified_7500' / 'training_report.json'
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
if not SELECTED_ARTIFACT.exists() or not TRAINING_REPORT.exists():
    raise FileNotFoundError('train.ipynb의 Router suite 학습을 먼저 완료하세요.')
training_report = json.loads(TRAINING_REPORT.read_text(encoding='utf-8'))
artifacts = {name: Path(row['artifact']) for name, row in training_report['variants'].items()}
selected_router = BudgetedUtilityRouter.load(SELECTED_ARTIFACT)
models = sorted({item.model_id for item in selected_router.assignments.values()})
if len(models) != 1:
    raise ValueError('Outcome matrix는 하나의 physical model을 사용해야 합니다.')
print('Selected backend:', training_report['selected_backend'])
print('Physical model:', models[0])

Selected backend: multitask_mlp
Physical model: deepseek/deepseek-v4-flash-0731


## 1. Frozen Test 8,337 manifest, materialization, candidate cache

In [3]:
index_report = ensure_frozen_index(config, mapping, progress=print)
cohort_report = build_cohort_manifests(config, cohort_config, output_directory=COHORT_DIR)
test_manifest = COHORT_DIR / 'cohort_test.jsonl'
materialization = materialize_dataset(
    config, mapping, output_directory=RUN_DIR / 'cases', splits=('test',),
    selection_manifests={'test': test_manifest}, progress=print,
)
candidate_summary = cache_candidates(
    RUN_DIR / 'cases' / 'cases_test.jsonl',
    RUN_DIR / 'candidates' / 'candidates_test.jsonl',
    max_source_bytes=config.max_source_bytes, parse_timeout_ms=config.parse_timeout_ms, progress=print,
)
print(json.dumps({'cohort': cohort_report['splits']['test'], 'materialization': materialization, 'candidates': candidate_summary}, ensure_ascii=False, indent=2))

materialize test: 1/8337
materialize test: 100/8337
materialize test: 200/8337
materialize test: 300/8337
materialize test: 400/8337
materialize test: 500/8337
materialize test: 600/8337
materialize test: 700/8337
materialize test: 800/8337
materialize test: 900/8337
materialize test: 1000/8337
materialize test: 1100/8337
materialize test: 1200/8337
materialize test: 1300/8337
materialize test: 1400/8337
materialize test: 1500/8337
materialize test: 1600/8337
materialize test: 1700/8337
materialize test: 1800/8337
materialize test: 1900/8337
materialize test: 2000/8337
materialize test: 2100/8337
materialize test: 2200/8337
materialize test: 2300/8337
materialize test: 2400/8337
materialize test: 2500/8337
materialize test: 2600/8337
materialize test: 2700/8337
materialize test: 2800/8337
materialize test: 2900/8337
materialize test: 3000/8337
materialize test: 3100/8337
materialize test: 3200/8337
materialize test: 3300/8337
materialize test: 3400/8337
materialize test: 3500/8337
mate

## 2. Full-5 measured outcome matrix 수집 (case당 API 최대 1회, 최대 1,000건 비동기 동시 처리)

In [4]:
selection_manifest = RUN_DIR / 'selections' / 'selected_test.jsonl'
outcome_path = RUN_DIR / 'outcomes' / 'outcomes_test.jsonl'
plan = plan_outcome_matrix(
    cases_path=RUN_DIR / 'cases' / 'cases_test.jsonl',
    candidate_cache=RUN_DIR / 'candidates' / 'candidates_test.jsonl',
    selection_manifest=selection_manifest, outcome_path=outcome_path, model_ids=models,
    max_candidates_per_case=MAX_CANDIDATES_PER_CASE, hard_negatives_per_case=HARD_NEGATIVES_PER_CASE,
)
print(json.dumps(plan, ensure_ascii=False, indent=2))
collection = collect_outcome_matrix(
    env_file=ENV_FILE, cases_path=RUN_DIR / 'cases' / 'cases_test.jsonl',
    candidate_cache=RUN_DIR / 'candidates' / 'candidates_test.jsonl',
    outcome_path=outcome_path, ledger_path=RUN_DIR / 'ledgers' / 'test_api_ledger.jsonl',
    model_ids=models, max_candidates_per_case=MAX_CANDIDATES_PER_CASE,
    hard_negatives_per_case=HARD_NEGATIVES_PER_CASE, max_concurrency=MAX_CONCURRENCY,
)
print(json.dumps(collection, ensure_ascii=False, indent=2))
if collection['status'] != 'complete':
    print('실패한 case만 남았습니다. 성공한 case는 저장되었으며 이 셀을 다시 실행하면 실패 case부터 재개합니다.')

{
  "selection_manifest": "D:\\llm-security\\Model_Evaluation\\work\\router_evaluation_full_test\\selections\\selected_test.jsonl",
  "selected_candidate_count": 14097,
  "positive_candidate_count": 5775,
  "hard_negative_candidate_count": 8322,
  "model": "deepseek/deepseek-v4-flash-0731",
  "assignment_count": 5,
  "expected_physical_api_requests": 8322,
  "completed_physical_api_requests": 0,
  "remaining_physical_api_requests": 8322,
  "expected_logical_expert_outcomes": 70485,
  "completed_logical_expert_outcomes": 0,
  "unexpected_existing_rows": 0,
  "outcome_path": "D:\\llm-security\\Model_Evaluation\\work\\router_evaluation_full_test\\outcomes\\outcomes_test.jsonl"
}
batched outcomes: 1/8322 completed; failed=0; request attempts this run=748
batched outcomes: 2/8322 completed; failed=0; request attempts this run=749
batched outcomes: 3/8322 completed; failed=0; request attempts this run=749
batched outcomes: 4/8322 completed; failed=0; request attempts this run=749
batched out

## 3. LR / GBDT / MLP 동일 Test 비교

In [5]:
expected_ids = list(selected_router.assignments)
audit = audit_outcome_matrix(outcome_path, expected_assignment_ids=expected_ids, selection_manifest=selection_manifest) if outcome_path.exists() else {'complete': False}
print(json.dumps(audit, ensure_ascii=False, indent=2))
evaluation_reports = {}
if audit['complete']:
    for backend, artifact in artifacts.items():
        evaluation_reports[backend] = evaluate_utility_router(
            artifact_path=artifact, test_outcomes=outcome_path,
            test_cases=RUN_DIR / 'cases' / 'cases_test.jsonl',
            candidate_cache=RUN_DIR / 'candidates' / 'candidates_test.jsonl',
            selection_manifest=selection_manifest,
            report_path=RESULT_DIR / f'evaluation_{backend}.json',
            max_candidates_per_case=MAX_CANDIDATES_PER_CASE,
        )
    print(json.dumps(evaluation_reports, ensure_ascii=False, indent=2))
else:
    print('Test outcome matrix가 아직 완성되지 않았습니다. 2번 셀을 다시 실행하세요.')

{
  "row_count": 70485,
  "candidate_group_count": 14097,
  "expected_assignment_count": 5,
  "duplicate_row_count": 0,
  "incomplete_candidate_group_count": 0,
  "incomplete_preview": [],
  "expected_candidate_group_count": 14097,
  "missing_candidate_group_count": 0,
  "missing_preview": [],
  "unexpected_candidate_group_count": 0,
  "complete": true
}
{
  "multitask_mlp": {
    "artifact": "D:\\llm-security\\Model_Evaluation\\artifacts\\juliet_utility_router_multitask_mlp.pkl",
    "test_outcomes": "D:\\llm-security\\Model_Evaluation\\work\\router_evaluation_full_test\\outcomes\\outcomes_test.jsonl",
    "test_matrix_audit": {
      "row_count": 70485,
      "candidate_group_count": 14097,
      "expected_assignment_count": 5,
      "duplicate_row_count": 0,
      "incomplete_candidate_group_count": 0,
      "incomplete_preview": [],
      "expected_candidate_group_count": 14097,
      "missing_candidate_group_count": 0,
      "missing_preview": [],
      "unexpected_candidate_group

## 4. 선택된 Router의 GT-matched finding 일괄 패치 평가

In [ ]:
if audit.get('complete'):
    verifier_commands = ([{'name': 'juliet-build', 'command': ['make'], 'timeout_seconds': 300.0}] if shutil.which('make') else [])
    patch_report = run_batched_patch_evaluation(
        env_file=ENV_FILE, detection_path=outcome_path.with_suffix('.detections.jsonl'),
        output_path=RUN_DIR / 'patches' / 'patch_results.jsonl',
        ledger_path=RUN_DIR / 'ledgers' / 'patch_api_ledger.jsonl',
        commands=verifier_commands, max_cases=PATCH_CASE_LIMIT,
    )
    patch_report['verification_mode'] = 'compile/test' if verifier_commands else 'apply-and-diff-check-only'
    (RESULT_DIR / 'patch_report.json').write_text(json.dumps(patch_report, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(patch_report, ensure_ascii=False, indent=2))

batched patches: 1 new cases; physical patch requests=0
batched patches: 2 new cases; physical patch requests=1
batched patches: 3 new cases; physical patch requests=1
batched patches: 4 new cases; physical patch requests=2
batched patches: 5 new cases; physical patch requests=3
batched patches: 6 new cases; physical patch requests=3
batched patches: 7 new cases; physical patch requests=3
batched patches: 8 new cases; physical patch requests=4
batched patches: 9 new cases; physical patch requests=4
batched patches: 10 new cases; physical patch requests=4
batched patches: 11 new cases; physical patch requests=4
batched patches: 12 new cases; physical patch requests=4
batched patches: 13 new cases; physical patch requests=4
batched patches: 14 new cases; physical patch requests=4
batched patches: 15 new cases; physical patch requests=4
batched patches: 16 new cases; physical patch requests=4
batched patches: 17 new cases; physical patch requests=4
batched patches: 18 new cases; physical 